In [3]:
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

In [4]:
# get the channel slug, episode guid, and transcription uuid for all transcriptions without a spaCy segmentation
transcripts = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            # check if any of the objects in segmentation_set have name == "spaCy"
            if podcast["language"].startswith("en") and not any(segmentation['name'] == "spaCy" for segmentation in transcription['segmentation_set']):
                transcripts.append({"slug": podcast['slug'], "ep_guid": audioitem['guid'], "trans_uuid": transcription['uuid']})
len(transcripts)

10

In [8]:
import sentence_splitter
import spacy
spacy_dict = {"en": "en_core_web_lg", "no": "nb_core_news_lg", "de": "de_dep_news_trf", "se": "sv_core_news_lg", "da": "da_core_news_trf"}

for transcript in transcripts:
    slug = transcript["slug"]
    ep_guid = transcript["ep_guid"]
    trans_uuid = transcript["trans_uuid"]
    lang = podcast["language"][0:2]

    # Get the text from the API
    data = requests.get(f"{SERVER}:{PORT}/api/transcriptions/{trans_uuid}/")
    transcript = data.json()
    try:
        # Split the text into sentences
        utterances = sentence_splitter.sentence_splitter(transcript, spacy_dict[lang])

        segmentation_dict = {
            "uuid": trans_uuid,
            "name": "spaCy",
            "segmentor": {"name": "spaCy", "version": spacy.__version__},
            "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
        }

        res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
        print(res.status_code)
    except Exception as e:
        print(e)
        print("SpaCy failed again", slug, ep_guid, lang )

local variable 'word' referenced before assignment
SpaCy failed again the-ben-shapiro-show 719502d2-eb58-11ed-afdd-2f453806c106 da
local variable 'word' referenced before assignment
SpaCy failed again the-ben-shapiro-show 299d0646-f3ff-11ed-984a-f704ed1b40be da
local variable 'word' referenced before assignment
SpaCy failed again the-daily 9c24aee6-c5ae-41e2-8ba7-c627630436f6 da
local variable 'word' referenced before assignment
SpaCy failed again the-daily 6154a385-4f53-46f8-8a1d-46d16dd2e316 da
local variable 'word' referenced before assignment
SpaCy failed again pod-save-america 9170cbd7-8a96-4189-ba86-819db6f8383d da
201
201
201
local variable 'word' referenced before assignment
SpaCy failed again ancient-health-podcast httpsapispreakercomepisode53832794 da
local variable 'word' referenced before assignment
SpaCy failed again ancient-health-podcast httpsapispreakercomepisode53777940 da


In [5]:
# copy a segmentation and write it back to the transcription with a new name

import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

target_segmentations = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    target_segmentations.append(segmentation)
target_segmentations


[{'transcription': 1,
  'uuid': '7909535e-dad4-11ed-ba56-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 2,
  'uuid': '7ad3aa86-dad4-11ed-980f-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 3,
  'uuid': '7cac1e2e-dad4-11ed-9e52-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 4,
  'uuid': '7e2630e6-dad4-11ed-9c3e-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 5,
  'uuid': '9a44cada-dad4-11ed-9644-00155d8020a1',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 106,
  'uuid': '8303cd26-dca5-11ed-bbbf-00155d0d192b',
  'name': 'spaCy',
  'segmentor': {'name': 'spaCy', 'version': '3.3.1'}},
 {'transcription': 107,
  'uuid': '1fecf512-dca7-11ed-878d-00155d0d192b',
  'name': 'spaCy',
  'segmentor': {'name': 'spaC

## FILTER TO NEW SEGMENTATION

In [3]:
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()

In [5]:
import numpy as np
import pandas as pd
import spacy
import requests

for podcast in podcasts[1:]:
    slug = podcast['slug']
    for audioitem in podcast['audioitem_set']:
        ep_guid = audioitem['guid']
        for transcription in audioitem['transcription_set']:
            trans_uuid = transcription['uuid']
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()

                    df = pd.DataFrame(seg["utterance_set"])
                    df["score"] = None

                    # get the "ClaimBuster-BBA-(COREF)" score for each utterance where it exists, otherwise use the "ClaimBuster-BBA" score
                    def get_score(classification_set):
                        score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA-(COREF)"), None)
                        if score is None:
                            score = next((float(cl["label"]) for cl in classification_set if cl["agent"] == "ClaimBuster-BBA"), None)
                            if score is None:
                                score = 0
                        return score
                    
                    df["score"] = df["classification_set"].apply(get_score)
                    # set all records that score below the 95th percentile to hidden
                    df.loc[df["score"] < df["score"].quantile(0.95), "visibility"] = 0
                    new_utterances = df.drop(columns=["score"]).to_dict(orient="records")


                    segmentation_dict = {
                        "uuid": trans_uuid,
                        "name": "spaCy-5% most CW",
                        "segmentor": {"name": "spaCy", "version": spacy.__version__},
                        "utterance_set": new_utterances
                    }
                    res = requests.post(f"{SERVER}:{PORT}/api/podcasts/{slug}/{ep_guid}/utterances/", json=segmentation_dict)
                    print(res.status_code)


201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201
201


Add new segmentation for Prolific trial run Transcription/Diarization/Advertising, with given podcast GUIDs

In [ ]:
guids = [
    "48723cca-0541-4efd-9f81-affe0046c365", # Ted Cruz, Hunter Biden Bombshells plus Rep. Jim Jordan Joins Us for Deep Dive on Biden Crime Family Part 1
    "01c17681-f1d1-4b4e-bac8-b000000ec0f8", # Ted Cruz, Deep Dive into Hunter Biden & the Weaponization of the Federal Government: Part 2 with Jim Jordan
    "8ecee10c-f011-11ed-aa53-f31ec92bee71", # Ben Shapiro, Ep. 1725 - Trump STEAMROLLS CNN
    "b6c46b88-ef48-11ed-8d23-db56bb3cb467", # Ben Shapiro, Ep. 1724 - Tucker's Back!
    "0319727e-0b17-4b00-b0ee-02da95c624ac", # Pod Save America, Trump’s CNN Clown Hall
    "ea7f5719-43e4-462a-bf94-3d6eb8b1b9d2", # Pod Save America, Tucker’s War With Fox
    "", # Al Franken, 
    "", # Al Franken, 
]

In [ ]:
# find the 5% of utterances with the highest average Checkworthiness score from both CB & FV
import numpy as np
import pandas as pd
df = pd.DataFrame(seg["utterance_set"])

# get the average Checkworthiness score for each utterance, from the label field of the classification objects
df["score"] = df["classification_set"].apply(lambda x: np.mean([float(cl["label"]) for cl in x if cl["category"] == "Checkworthy" and (cl["agent"] == "ClaimBuster-BBA" or cl["agent"] == "Factiverse")]))
